# 现役日频输入与冻结推理核查
原模型身份固定，验证39项主输入及公共分支，不重新拟合，不把旧输入冒充新决策。

In [1]:
import os,json,hashlib,pickle
from pathlib import Path
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.workflow.direct_continuous_soxx_allocation import _load_frame,DirectContinuousAllocationSpec
from src.workflow.pit_constituent_auxiliary_training_allocation import _soxx_common_training_frame,PitAuxiliaryAllocationSpec
from src.workflow.minute_pit_allocator import _frozen_predictions
from src.data.qlib_native import QlibProviderConfig,load_qlib_feature_frame
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
card=root/'research/daily_input_readiness_20261014';plan=json.loads((card/'plan.json').read_text())
release=root/'production/stage1_futu_sim_v1/runtime_release.json'
assert hashlib.sha256(release.read_bytes()).hexdigest()==plan['release_sha256']
providers=['soxl','soxl_soxx_soxs','dynamic_soxx_soxl_cash_allocation_wf_v1']
identities={name:hashlib.sha256((root/'data/qlib'/name/'qlib_store_manifest.json').read_bytes()).hexdigest() for name in providers}
assert identities==plan['provider_manifest_sha256']
qlib.init(provider_uri=str(root/'data/qlib/soxl'),region='us')
frame,audit,features=_load_frame(DirectContinuousAllocationSpec(root=root),include_incomplete_latest=True)
common=_soxx_common_training_frame(PitAuxiliaryAllocationSpec(root=root),end=plan['as_of'])
parent=root/plan['parent_card'];bundle=json.loads((parent/'frozen_bundle_manifest.json').read_text())
item=next(a for a in bundle['artifacts'] if a['path'].endswith('models/pit_only_aux_005.pkl'))
assert item['sha256']==plan['model_sha256']
with R.uri_context(str(parent/'qlib_recorder')):
    original=R.get_recorder(recorder_id=plan['model_recorder_id'],experiment_name=parent.name)
    path=Path(original.download_artifact(item['path'].split('/artifacts/',1)[1]))
    assert hashlib.sha256(path.read_bytes()).hexdigest()==item['sha256']
    models=pickle.loads(path.read_bytes())
state=models['outer_refit'];assert state['primary_features']==features and len(features)==39
complete=np.isfinite(frame[features]).all(axis=1)&np.isfinite(common[state['common_features']].reindex(frame.index)).all(axis=1)
latest=frame.index[complete].max();selected=frame.loc[[latest],features]
parent_plan=json.loads((root/'research/minute_information_pit_allocator_20261012/plan.json').read_text())
pred=_frozen_predictions({'frozen_models':models,'features':selected,'common':common.reindex(selected.index)},window='outer',plan=parent_plan)
assert np.isfinite(pred.to_numpy()).all() and np.allclose(pred[['soxl_weight','soxx_weight','cash_weight']].sum(axis=1),1.)
import exchange_calendars as xc
cal=xc.get_calendar('XNYS');decision_date=cal.next_session(latest).date().isoformat()
# Check observed same-date prices using native Qlib sources on both sides.
daily=load_qlib_feature_frame(QlibProviderConfig(root/'data/qlib/soxl'),instruments=['SOXL','SOXX'],fields=['$open','$close'],start='2026-10-01',end=plan['as_of']).reset_index()
minute=load_qlib_feature_frame(QlibProviderConfig(root/'data/qlib/nas_minute_features_20261014/core',freq='1min'),instruments=['SOXL','SOXX'],fields=['$open','$close'],start='2026-10-01',end=plan['as_of']+' 16:00:00').reset_index()
for data in (daily,minute):data['date']=data.datetime.dt.normalize()
aggregated=minute.groupby(['instrument','date']).agg(minute_open=('$open','first'),minute_close=('$close','last'))
comparison=daily.set_index(['instrument','date']).join(aggregated)
comparison['open_difference']=comparison['$open']-comparison.minute_open
comparison['close_difference']=comparison['$close']-comparison.minute_close
assert comparison[['open_difference','close_difference']].abs().max().max()<1e-6
rows=[]
for name in providers:
    manifest=json.loads((root/'data/qlib'/name/'qlib_store_manifest.json').read_text())
    rows.append({'provider':name,'as_of':manifest['as_of'],'calendar_end':manifest['calendar_end'],'manifest_sha256':identities[name]})
summary={'experiment':card.name,'as_of':plan['as_of'],'provider_uri':str(root/'data/qlib/soxl'),'region':'us','source_id':plan['source_id'],'primary_feature_count':len(features),'common_feature_count':len(state['common_features']),'latest_complete_signal':str(latest.date()),'latest_supported_decision_date':decision_date,'requested_decision_date':'2026-10-14','primary_missing_at_latest':int(frame.loc[latest,features].isna().sum()),'paired_price_rows':len(comparison),'maximum_price_difference':float(comparison[['open_difference','close_difference']].abs().max().max()),'new_fits':0,'new_backtests':0,'production_unchanged':True,'objective_achieved':False,'decision':'frozen_inference_verified_daily_inputs_stale_for_oct14','model_recorder_id':plan['model_recorder_id'],'model_sha256':plan['model_sha256'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/data.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*']}}
assert summary['latest_supported_decision_date']<'2026-10-14'
assert hashlib.sha256(release.read_bytes()).hexdigest()==plan['release_sha256']
assert identities=={name:hashlib.sha256((root/'data/qlib'/name/'qlib_store_manifest.json').read_bytes()).hexdigest() for name in providers}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='daily_input_readiness',uri=str(tracking)):
    rec=R.get_recorder();rec.log_params(model_sha256=plan['model_sha256'],new_fits=0,new_backtests=0)
    rec.save_objects(artifact_path='inputs',plan=plan,source_audit=audit,provider_status=pd.DataFrame(rows),primary_latest=selected,common_latest=common.loc[selected.index,state['common_features']])
    rec.save_objects(artifact_path='analysis',summary=summary,frozen_prediction=pred,price_comparison=comparison)
    artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(primary_feature_count=39,registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking))
atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json')
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id))
atomic_json(summary,card/'summary.json');pd.DataFrame(rows).to_csv(card/'provider_status.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':summary['provider_uri'],'recorder_uri':str(tracking),'region':'us','as_of':plan['as_of'],'artifact_policy':summary['artifact_policy']},sort_keys=False))
print({k:summary[k] for k in ['latest_complete_signal','latest_supported_decision_date','common_feature_count','maximum_price_difference','recorder_id']})


In [1]:
from pathlib import Path
import json,hashlib,os
from datetime import datetime
import qlib
from qlib.workflow import R
from src.data.market_update import _max_settled_us_daily_date
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
card=root/'research/daily_input_readiness_20261014'
summary=json.loads((card/'summary.json').read_text())
observed=datetime.fromisoformat('2026-10-13T22:26:43+00:00')
policy={'observed_at':observed.isoformat(),'maximum_settled_daily_date':_max_settled_us_daily_date(now=observed).isoformat(),'configured_settlement_clock':'20:15 America/New_York','next_settlement_at':'2026-10-13T20:15:00-04:00','requested_decision_date':'2026-10-14','daily_source_on_schedule_at_observation':True,'future_decision_ready':False}
assert policy['maximum_settled_daily_date']==summary['latest_complete_signal']
summary['decision']='frozen_inference_verified_pending_daily_settlement_for_oct14'
summary['daily_settlement_policy']=policy
qlib.init(provider_uri=summary['provider_uri'],region='us')
with R.uri_context(summary['recorder_uri']):
    rec=R.get_recorder(recorder_id=summary['recorder_id'],experiment_name=card.name)
    rec.save_objects(artifact_path='analysis',summary=summary,daily_settlement_policy=policy)
    rec.save_objects(artifact_path='sources',daily_settlement_code=(root/'src/data/market_update.py').read_text())
    artifacts=Path(summary['recorder_uri'])/summary['experiment_id']/summary['recorder_id']/'artifacts'
    rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json')
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=summary['experiment_id'])
atomic_json(summary,card/'summary.json');atomic_json(policy,card/'daily_settlement_policy.json')
print(policy)


# 冻结模型推理通过，等待既有日线结算时点

核查现役f696f656模型，未重新训练、未回测。39个主输入及16个公共分支输入在2026-10-12完整，冻结模型原始目标推理有限且三资产权重和为1。推理对象由原Qlib Recorder读取，并与固定SHA256校验一致；原始目标不等于Stage1动作层最终订单，保留在本卡Recorder，不生成交易指令。

10月1日至12日共16个标的日，现有日频SOXL/SOXX开收盘价与NAS分钟Qlib聚合值差异为0。此检查仅覆盖该窗口开收盘价，不认证全部历史、成交量、VIX或其他标的。

关键时点：核查时间为2026-10-13 22:26:43 UTC，即纽约18:26。仓库`_max_settled_us_daily_date`要求纽约20:15后才使用当日日线；此刻最大可结算日为10月12日。因此三份日频provider停在10月12日符合当前政策，并非已确认的刷新故障。

10月12日输入支持10月13日决策，不支持10月14日的新决策。分钟源到10月13日不能替代缺少的其他日频来源。下一结算时点为纽约10月13日20:15（北京时间10月14日08:15）；当前未声称有刷新进程在运行，也没有提前放宽结算门禁。

决定：**frozen_inference_verified_pending_daily_settlement_for_oct14**。完整输入更新仍未完成，0拟合、0回测，无新的收益或稳定超额证据；现役release、模型参数及原始provider均未修改。历史和近期收益门禁没有在本卡重新评估。

下一步在结算时间到达后核查既有日频刷新结果，确认VIX、跨资产、SOXL/SOXX及公共分支共同覆盖10月13日，再做10月14日推理。分钟特征已被否决的研究路线保持关闭，不以新增两天数据启动参数搜索。

Recorder `f649858ca76a49afad5d4921767455fa`。[来源日期](provider_status.csv) · [结算时点](daily_settlement_policy.json) · [证据清单](verification_manifest.json)
